In [ ]:
# import data from notebook 02

import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("../data/interim/02_adata_parsedPerturb.h5ad")
print(adata)

In [ ]:
# check efficacy of CRISPRa
# look at target gene's expr from lognorm across cells w that guide vs ctrl cells

import numpy as np

def get_gene_expr(adata, gene, layer="lognorm"):
    """Pull one gene's expression vector across all cells as a flat array."""

    # adata.var_names stores the idx (genes) ordered same as in adata.X (the actual sparse data matrix)
    idx = adata.var_names.get_loc(gene) # find col idx (int) for gene (basically which col # is this gene in adata.X)
    
    # taking the lognorm layer, turning adata.X and dropping all cols except gene of interest
    # turning sparse matrix into np array (writes 0s in for missing values in sparse)
    return np.asarray(adata.layers[layer][:, idx].todense()).flatten()

# quick test on one target gene present in our single-perturbation set
test_gene = "ARID1A"
expr = get_gene_expr(adata, test_gene)
is_target = (adata.obs["target"] == test_gene).values
is_control = (adata.obs["target"] == "control").values

print("target cells mean expr:", expr[is_target].mean())
print("control cells mean expr:", expr[is_control].mean())

# the direction of the change is correct, but modest effect. this may or may not be a flag. test more systematically below

In [ ]:
# compute on-target fold-change and siginificance for all 105 targets at once

from scipy import stats

results = []
for gene in adata.obs.loc[adata.obs["perturbation_type"] == "single", "target"].unique():
    if gene not in adata.var_names:
        continue  # target gene itself not detected/retained in the matrix
    # get expression array for the gene
    expr = get_gene_expr(adata, gene)
    is_target = (adata.obs["target"] == gene).values
    is_control = (adata.obs["target"] == "control").values

    mean_target = expr[is_target].mean()
    mean_control = expr[is_control].mean()
    stat, pval = stats.mannwhitneyu(expr[is_target], expr[is_control])

    results.append({
        "gene": gene, "n_cells": is_target.sum(),
        "mean_target": mean_target, "mean_control": mean_control,
        "log2fc": np.log2((mean_target + 0.01) / (mean_control + 0.01)),
        "pval": pval
    })

onTarget = pd.DataFrame(results).sort_values("log2fc", ascending=False)
onTarget.head(10)

In [ ]:
# because so many pvals are 0, there will be issues with volcano and other plots

frac_above = []
for gene in onTarget["gene"]:
    expr = get_gene_expr(adata, gene) # our fxn returning nparray for gene from sparse matrix, all cells
    is_target = (adata.obs["target"] == gene).values # get metadata for cells that were targeted for this gene
    is_control = (adata.obs["target"] == "control").values # get metadata for controls (not targeted for any gene)
    control_mean = expr[is_control].mean()
    # because expr is a nparray, using > triggers comparing every value to control_mean and returning T/F
    # that is a bool list, then you do mean on it (so you get pct targeted cells that are above ctrl levels for gene)
    frac_above.append((expr[is_target] > control_mean).mean())

onTarget["frac_cells_above_control_mean"] = frac_above
print(onTarget.columns.tolist())

In [ ]:
# find the median gene in the perturbations - with median fold change

# top 2 by log2fc, and 2 genes closest to the screen's median log2fc
median_log2fc = onTarget["log2fc"].median()
onTarget["dist_from_median"] = (onTarget["log2fc"] - median_log2fc).abs()

top2 = onTarget.sort_values("log2fc", ascending=False).head(2)["gene"].tolist()
median2 = onTarget.sort_values("dist_from_median").head(2)["gene"].tolist()
example_genes = top2 + median2
print(example_genes)

In [ ]:
# figure showing cells and gene counts, by perturbed gene

from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))

# Panel 1 (unchanged)
ax0 = axes[0]
ax0.hist(onTarget["log2fc"], bins=30, color="#8172B2", edgecolor="white", alpha=0.85)
ax0.axvline(0, color="grey", linestyle="--", linewidth=1)
ax0.axvline(onTarget["log2fc"].median(), color="black", linestyle="-", linewidth=1,
            label=f"median = {onTarget['log2fc'].median():.2f}")
ax0.set_xlim(-6, onTarget["log2fc"].max() + 0.5)
ax0.set_xticks(list(range(-6, int(onTarget["log2fc"].max()) + 2, 2)))
ax0.set_xlabel("log2 fold change (target vs. control)")
ax0.set_ylabel("number of target genes")
ax0.set_title("A. Distribution of on-target log2FC")
ax0.legend()

# Panel 2 (unchanged)
ax1 = axes[1]
top_genes = onTarget.sort_values("log2fc", ascending=False).head(15)["gene"].tolist()
n_jitter_points = 150
rng = np.random.default_rng(0)
bar_width = 0.35
x = np.arange(len(top_genes))
target_means, target_sems, control_means, control_sems = [], [], [], []

for i, gene in enumerate(top_genes):
    expr = get_gene_expr(adata, gene)
    is_target = (adata.obs["target"] == gene).values
    is_control = (adata.obs["target"] == "control").values
    tvals, cvals = expr[is_target], expr[is_control]
    target_means.append(tvals.mean())
    target_sems.append(tvals.std(ddof=1) / np.sqrt(len(tvals)))
    control_means.append(cvals.mean())
    control_sems.append(cvals.std(ddof=1) / np.sqrt(len(cvals)))
    t_sub = rng.choice(tvals, size=min(n_jitter_points, len(tvals)), replace=False)
    c_sub = rng.choice(cvals, size=min(n_jitter_points, len(cvals)), replace=False)
    jitter_t = rng.normal(x[i] - bar_width/2, 0.05, size=len(t_sub))
    jitter_c = rng.normal(x[i] + bar_width/2, 0.05, size=len(c_sub))
    ax1.scatter(jitter_t, t_sub, s=4, color="#1f3d7a", alpha=1, zorder=3)
    ax1.scatter(jitter_c, c_sub, s=4, color="#a8c4e8", alpha=1, zorder=3)

ax1.bar(x - bar_width/2, target_means, bar_width, yerr=target_sems, capsize=3,
        color="#1f3d7a", alpha=0.65, label="CRISPRa target", zorder=2)
ax1.bar(x + bar_width/2, control_means, bar_width, yerr=control_sems, capsize=3,
        color="#a8c4e8", alpha=0.5, label="control", zorder=2)
ax1.set_xticks(x)
ax1.set_xticklabels(top_genes, rotation=45, ha="right")
ax1.set_ylabel("lognorm expression")
ax1.set_title("B. Top 15 targets: expression vs. control")
ax1.legend()

# Panel 3: smoothed density curves (KDE) instead of histograms, for the same
# top-2 + median-2 genes + control. Y-axis is capped just above the tallest
# TARGET curve -- control's density spikes far higher near zero (since most
# cells express near-baseline) and would otherwise squash the target curves
# flat. A break mark on the y-axis signals the control curve is clipped, and
# each group's n is annotated directly since it's no longer readable from bar height.
ax2 = axes[2]
gene_colors = ["#C44E52", "#DD8452", "#4C72B0", "#55A868"]
xgrid = np.linspace(0, 4, 300)

is_control = (adata.obs["target"] == "control").values
control_expr = get_gene_expr(adata, example_genes[0])[is_control]
control_kde = gaussian_kde(control_expr)(xgrid)
ax2.plot(xgrid, control_kde, color="grey", linewidth=2, label=f"control (n={is_control.sum()})")

target_curve_maxes = []
for gene, color in zip(example_genes, gene_colors):
    expr = get_gene_expr(adata, gene)
    is_target = (adata.obs["target"] == gene).values
    tvals = expr[is_target]
    kde_vals = gaussian_kde(tvals)(xgrid)
    target_curve_maxes.append(kde_vals.max())
    ax2.plot(xgrid, kde_vals, color=color, linewidth=2, label=f"{gene} (n={is_target.sum()})")

# cap y-axis just above tallest target curve, clipping control's peak
y_cap = max(target_curve_maxes) * 1.15
ax2.set_ylim(0, y_cap)

# break-symbol marks on the y-axis to signal truncation
d = 0.015
kwargs = dict(transform=ax2.transAxes, color="k", clip_on=False, linewidth=1)
ax2.plot((-d, +d), (0.97, 1.03), **kwargs)
ax2.plot((-d + 0.01, +d + 0.01), (0.97, 1.03), **kwargs)

ax2.set_xlabel("lognorm expression")
ax2.set_ylabel("density (control curve clipped, see break)")
ax2.set_title("C. Expression distributions:\ntop (HES7, COL1A1) and median (S1PR, BAK1) performers")
ax2.legend(fontsize=8)

fig.suptitle("On-target CRISPRa efficacy across 105 single-gene perturbations", fontsize=13)
fig.tight_layout()
fig.savefig("../results/figures/on_target_efficacy.png", dpi=150, bbox_inches="tight")
plt.show()

Figure 1. On-target CRISPRa activation efficacy across the Norman et al. 2019 single-gene perturbation screen. Analysis restricted to the 102/105 single-gene CRISPRa targets present in the filtered gene set (n = 102,337 cells post-QC; guide assignment required good_coverage == True). For each target gene, log-normalized (CP10K, log1p) expression of that gene itself was compared between cells carrying its activating guide and the shared non-targeting control population (n = 10,990 cells).

(A) Distribution of log2 fold-change (target vs. control) across all single-gene targets. The distribution is overwhelmingly shifted positive (median log2FC = 2.08), consistent with the expected direction of effect for CRISPRa — the large majority of guides successfully increased expression of their intended target.

(B) Mean lognorm expression (± SEM) for the 15 strongest-activating targets, target vs. control, with a jittered subsample (n = 150 cells/group) overlaid to show within-group spread. Bar and plungers understate cell-to-cell heterogeneity in activation of target.

(C) Kernel density estimates of expression for the two strongest-activating genes and two genes closest to the screen's median log2FC, against the control distribution. Control's y-axis is clipped. Group sizes (n) are reported in the legend rather than encoded in curve height. Note that "median performer" genes show more cells with 0 change in target.

Note: statistical significance (Mann-Whitney U) was conducted; p ~0 for the majority of targets (75/102 genes had p = 0) irrespective of L2FC - not useful in ranking genes, here.

In [ ]:
from scipy.stats import theilslopes

# Panel A: guide UMI count (per cell) for target cells (only needs to be computed once)
guide_umi_mean = []
for gene in onTarget["gene"]:
    is_target = (adata.obs["target"] == gene).values
    guide_umi_mean.append(adata.obs.loc[is_target, "UMI_count"].mean())
onTarget["mean_guide_UMI"] = guide_umi_mean

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# Panel A: penetrance histogram with gene names listed inside each bar, comma-separated
ax0 = axes[0]
bins = np.linspace(0, 1, 21)
counts, bin_edges, patches = ax0.hist(
    onTarget["frac_cells_above_control_mean"], bins=bins,
    color="#4C72B0", edgecolor="white", alpha=0.85
)
ax0.axvline(onTarget["frac_cells_above_control_mean"].median(), color="black",
            linewidth=1, label=f"median = {onTarget['frac_cells_above_control_mean'].median():.2f}")

bin_idx = np.digitize(onTarget["frac_cells_above_control_mean"], bin_edges) - 1
for i, patch in enumerate(patches):
    genes_in_bin = onTarget.loc[bin_idx == i, "gene"].tolist()
    if not genes_in_bin:
        continue
    bar_x = patch.get_x() + patch.get_width() / 2
    bar_height = patch.get_height()
    label = ", ".join(genes_in_bin)
    ax0.text(bar_x, bar_height * 0.98, label, rotation=90, color="white",
              fontsize=5, ha="center", va="top")

ax0.set_xlabel("fraction of target cells above control mean")
ax0.set_ylabel("number of target genes")
ax0.set_title("A. Penetrance across all single-gene targets")
ax0.legend()

# Panels B & C: log-x scatter with robust (Theil-Sen) fit, dashed, matching scatter color
for ax, xcol, xlabel, color, title in zip(
    axes[1:],
    ["n_cells", "mean_guide_UMI"],
    ["number of cells for this target (log scale)", "mean guide barcode UMI count (log scale)"],
    ["#55A868", "#C44E52"],
    ["B. Effect size vs. sample size (log x, robust fit)",
     "C. Guide dosage vs. on-target efficacy (log x, robust fit)"]
):
    x = onTarget[xcol].values
    y = onTarget["log2fc"].values
    ax.scatter(x, y, s=15, alpha=0.6, color=color)
    ax.set_xscale("log")

    ts_slope, ts_intercept, _, _ = theilslopes(y, np.log10(x))
    xline = np.linspace(np.log10(x.min()), np.log10(x.max()), 50)
    ax.plot(10**xline, ts_slope * xline + ts_intercept, color=color,
            linestyle="--", linewidth=1.5, label=f"Theil-Sen slope={ts_slope:.2f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("log2 fold change")
    ax.set_title(title)
    ax.legend()

fig.suptitle("Screen-wide characterization: penetrance, power, and guide dosage", fontsize=13)
fig.tight_layout()
fig.savefig("../results/figures/screen_characterization.png", dpi=150, bbox_inches="tight")
plt.show()


Figure 2. Screen-wide penetrance, statistical power, and guide dosage effects.

(A) Distribution of penetrance (fraction of a target's cells with expression above the control mean) across all 102 single-gene targets. Most targets show penetrance well below 1.0, confirming that a responder/non-responder split is warranted before downstream (perturbed vs ctrl-guide cells) DE analysis.

(B) L2FC vs. number of cells recovered per target gene (log-scaled x-axis, robust Theil-Sen fit). A positive slope (≈1 per log10 unit of cell count) indicates targets with more recovered cells tend to show larger L2FC - which suggests the data is robust (usually undersampling can lead to inflated L2FC)

(C) L2FC vs. mean guide-barcode UMI count per target cell (log-scaled x-axis, robust Theil-Sen fit). A positive slope (≈0.95) indicates that cells with more detected guide transcript show stronger target activation